# Code Generator (Trình sinh mã)

Yêu cầu: dùng Frontier model (mô hình hàng đầu) để sinh mã C++ hiệu năng cao từ mã Python


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Nhắc nhở: TÙY CHỌN khi thực thi mã C++</h2>
            <span style="color:#f71;">Cách khác: bạn có thể chạy trên website đã giới thiệu hôm qua</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Lưu ý quan trọng</h1>
            <span style="color:#900;">
            Trong lab (bài thực hành) này, mình dùng các model mã nguồn mở miễn phí trên Ollama. Mình cũng dùng các model mã nguồn mở trả phí qua Groq và OpenRouter. Chỉ chọn những model bạn muốn!
            </span>
        </td>
    </tr>
</table>

In [ ]:
# imports (các thư viện)

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess


In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key tồn tại và bắt đầu bằng {openai_api_key[:8]}")
else:
    print("Chưa thiết lập OpenAI API Key")
    
if anthropic_api_key:
    print(f"Anthropic API Key tồn tại và bắt đầu bằng {anthropic_api_key[:7]}")
else:
    print("Chưa thiết lập Anthropic API Key (và đây là tùy chọn)")

if google_api_key:
    print(f"Google API Key tồn tại và bắt đầu bằng {google_api_key[:2]}")
else:
    print("Chưa thiết lập Google API Key (và đây là tùy chọn)")

if grok_api_key:
    print(f"Grok API Key tồn tại và bắt đầu bằng {grok_api_key[:4]}")
else:
    print("Chưa thiết lập Grok API Key (và đây là tùy chọn)")

if groq_api_key:
    print(f"Groq API Key tồn tại và bắt đầu bằng {groq_api_key[:4]}")
else:
    print("Chưa thiết lập Groq API Key (và đây là tùy chọn)")

if openrouter_api_key:
    print(f"OpenRouter API Key tồn tại và bắt đầu bằng {openrouter_api_key[:6]}")
else:
    print("Chưa thiết lập OpenRouter API Key (và đây là tùy chọn)")



In [ ]:
# Kết nối các client library (thư viện client)

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
grok_url = "https://api.x.ai/v1"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)


In [ ]:
models = ["gpt-5", "claude-sonnet-4-5-20250929", "grok-4", "gemini-2.5-pro", "qwen2.5-coder", "deepseek-coder-v2", "gpt-oss:20b", "qwen/qwen3-coder-30b-a3b-instruct", "openai/gpt-oss-120b", ]

clients = {"gpt-5": openai, "claude-sonnet-4-5-20250929": anthropic, "grok-4": grok, "gemini-2.5-pro": gemini, "openai/gpt-oss-120b": groq, "qwen2.5-coder": ollama, "deepseek-coder-v2": ollama, "gpt-oss:20b": ollama, "qwen/qwen3-coder-30b-a3b-instruct": openrouter}

# Muốn giữ chi phí cực thấp? Thay bằng các model bạn chọn, dùng ví dụ từ hôm qua

In [ ]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

## Ghi đè các lệnh này bằng lệnh từ hôm qua

Hoặc dùng website như hôm qua:

 https://www.programiz.com/cpp-programming/online-compiler/

In [ ]:
compile_command = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]


## Tiếp theo: nhiệm vụ chính

In [ ]:
system_prompt = """
Nhiệm vụ của bạn là chuyển mã Python thành mã C++ hiệu năng cao (high performance).
Chỉ trả lời bằng mã C++. Không giải thích, trừ một vài comment (chú thích) khi cần.
Mã C++ phải cho ra output (kết quả in ra) giống hệt, trong thời gian ngắn nhất có thể.
"""

def user_prompt_for(python):
    return f"""
Port (chuyển) mã Python này sang C++ với implementation (cách hiện thực) nhanh nhất, cho ra output giống hệt trong thời gian ngắn nhất.
Thông tin hệ thống là:
{system_info}
Phản hồi của bạn sẽ được ghi vào file tên main.cpp rồi biên dịch và thực thi; lệnh compilation (biên dịch) là:
{compile_command}
Chỉ trả lời bằng mã C++.
Mã Python cần port:

```python
{python}
```
"""

In [ ]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [ ]:
def write_output(cpp):
    with open("main.cpp", "w") as f:
        f.write(cpp)

In [ ]:
def port(model, python):
    client = clients[model]
    openai_reasoning_models = {"gpt-5"}
    reasoning_effort = "high" if model in openai_reasoning_models else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)
    return reply

In [ ]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Kết quả (Result): {result:.12f}")
print(f"Thời gian thực thi (Execution Time): {(end_time - start_time):.6f} giây")
"""

In [ ]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Lỗi (Error): {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [ ]:
def compile_and_run():
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    except subprocess.CalledProcessError as e:
        print(f"Đã xảy ra lỗi (error):\n{e.stderr}")

In [ ]:
with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label="Mã Python:", lines=28, value=pi)
        cpp = gr.Textbox(label="Mã C++:", lines=28)
    with gr.Row():
        model = gr.Dropdown(models, label="Chọn model", value=models[0])
        convert = gr.Button("Chuyển mã")

    convert.click(port, inputs=[model, python], outputs=[cpp])

ui.launch(inbrowser=True)

In [ ]:
compile_and_run()

Qwen 2.5 Coder: Fail (thất bại)  
DeepSeek Coder v2: 0.114050084  
OpenAI gpt-oss 20B: 0.080438  
Qwen 30B: 0.113734  
OpenAI gpt-oss 120B: 1.407383




Trong thí nghiệm của Ed, các mức tăng tốc (performance speedup) là:

Hạng 9: Qwen 2.5 Coder: Fail (thất bại)  
Hạng 8: OpenAI GPT-OSS 120B: 14X speedup (tăng tốc)    
Hạng 7: DeepSeek Coder v2: 168X speedup (tăng tốc)  
Hạng 6: Qwen3 Coder 30B: 168X speedup (tăng tốc)   
Hạng 5: Claude Sonnet 4.5: 184X speedup (tăng tốc)   
Hạng 4: GPT-5: 233X speedup (tăng tốc)  
**Hạng 3: oss-20B: 238X speedup (tăng tốc)**  
Hạng 2: Grok 4: 1060X speedup (tăng tốc)  
Hạng 1: Gemini 2.5 Pro: 1440X speedup (tăng tốc)  